# PSD rounding controls

This companion needs only NumPy, JAX and Jupyter. It holds one boundary fixed at a time: Hann window, segment spectrum, complex power, averaging, bias division and final reciprocal. It demonstrates generic floating-point arithmetic; it does not implement or validate a complete PyCBC PSD estimator.

For original PyCBC/LAL implementations and exact validation switches, see the [PSD comparison notebook](jax_psd_numerical_differences.ipynb). Differences printed here depend on the displayed libraries and device and may be zero. The two supplied power expansions are deliberately unequal even though their exact-arithmetic formulas agree.

In [1]:
import sys
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
try:
    devices = jax.devices("gpu")
except RuntimeError:
    devices = jax.devices("cpu")
device = devices[0]
print({"Python": sys.version.split()[0], "NumPy": np.__version__,
       "JAX": jax.__version__, "platform": device.platform,
       "device": device.device_kind, "x64": jax.config.jax_enable_x64})

def on_device(values):
    return jax.device_put(values, device)

def report(name, left, right):
    left, right = np.asarray(left), np.asarray(right)
    wide = np.complex128 if np.iscomplexobj(left) else np.float64
    difference = np.abs(left.astype(wide) - right.astype(wide))
    print(f"{name}: differing values={np.count_nonzero(left != right)}/{left.size}; "
          f"max absolute difference={float(np.max(difference)):.9g}")


{'Python': '3.13.3', 'NumPy': '2.2.5', 'JAX': '0.11.2', 'platform': 'gpu', 'device': 'NVIDIA GeForce RTX 3090', 'x64': True}


## Hold the Hann window while comparing its energy

Window generation uses cosine evaluations and arithmetic that can round differently. Casting to float32 can remove differences visible in float64. Passing the same NumPy window to both energy sums separates window construction from squaring and reduction.

In [2]:
for dtype in (np.float32, np.float64):
    common_window = np.hanning(64).astype(dtype)
    with jax.default_device(device):
        generated_window = np.asarray(jnp.hanning(64).astype(dtype))
    report(dtype.__name__ + " generated Hann", common_window, generated_window)
    cpu_energy = np.sum(common_window * common_window, dtype=dtype)
    device_energy = np.asarray(jax.jit(lambda w: jnp.sum(w * w))(
        on_device(common_window)))
    print("  held-window energy:", float(cpu_energy), float(device_energy))


float32 generated Hann: differing values=0/64; max absolute difference=0
  held-window energy: 23.625 23.625
float64 generated Hann: differing values=40/64; max absolute difference=3.33066907e-16
  held-window energy: 23.625 23.625


## Hold a windowed segment, then hold its spectrum

The same float32 segment enters NumPy and JAX real Fourier transforms. Their outputs can differ because of intermediate precision and transform ordering. An output comparison does not reveal the unique internal butterfly responsible.

Next, one common complex64 spectrum enters both power expressions, separating power rounding from the preceding FFT. The NumPy expression is `abs(z * conjugate(z))`; the JAX expression is its real part. These are arithmetic controls, not substitutes for the original PSD implementation.

In [3]:
rng = np.random.default_rng(1729)
held_segment = (rng.standard_normal(64).astype(np.float32)
                * np.hanning(64).astype(np.float32))
numpy_spectrum = np.fft.rfft(held_segment).astype(np.complex64)
device_spectrum = np.asarray(jax.jit(jnp.fft.rfft)(on_device(held_segment)))
report("Held-segment FFT (complex64 outputs)", numpy_spectrum, device_spectrum)

held_spectrum = numpy_spectrum.copy()
numpy_power = np.abs(held_spectrum * held_spectrum.conj())
device_power = np.asarray(jax.jit(lambda z: jnp.real(z * jnp.conj(z)))(
    on_device(held_spectrum)))
report("Held-spectrum power", numpy_power, device_power)


Held-segment FFT (complex64 outputs): differing values=30/33; max absolute difference=1.47453748e-06
Held-spectrum power: differing values=3/33; max absolute difference=3.81469727e-06


## Isolate the complex-power rounding boundary

Float32 components have exact binary64 squares. Retaining the real square until the addition differs from first rounding both squares to float32. The supplied complex64 values make this distinction visible. The fingerprints below report which expansion matches each displayed library; they do not identify a unique machine instruction on every compiler.

`abs(z)**2` is a separate reordered formula with its own magnitude rounding, not either expression above.

In [4]:
power_inputs = np.array([
    complex(6.472284439951181e-6, 5.454407073557377e-6),
    complex(0.0728302001953125, -0.00025936285965144634)
], dtype=np.complex64)
real, imaginary = power_inputs.real, power_inputs.imag
rounded_real_square = np.float32(real * real)
rounded_imaginary_square = np.float32(imaginary * imaginary)
separate_squares = np.float32(rounded_real_square + rounded_imaginary_square)
retained_real_square = np.float32(
    real.astype(np.float64) ** 2 + rounded_imaginary_square.astype(np.float64))
assert np.all(separate_squares != retained_real_square)

numpy_expression = np.abs(power_inputs * power_inputs.conj())
device_expression = np.asarray(jax.jit(lambda z: jnp.real(z * jnp.conj(z)))(
    on_device(power_inputs)))
print("Separately rounded squares:", separate_squares)
print("Real square retained:", retained_real_square)
print("NumPy abs(z*conj(z)):", numpy_expression)
print("JAX real(z*conj(z)):", device_expression)
print("NumPy matches retained-real control:",
      numpy_expression.tobytes() == retained_real_square.tobytes())
print("NumPy matches separate-squares control:",
      numpy_expression.tobytes() == separate_squares.tobytes())
print("JAX matches separate-squares control:",
      device_expression.tobytes() == separate_squares.tobytes())
print("JAX matches retained-real control:",
      device_expression.tobytes() == retained_real_square.tobytes())
print("Separate abs(z)**2 control:", np.float32(np.abs(power_inputs) ** 2))


Separately rounded squares: [7.1641026e-11 5.3043049e-03]
Real square retained: [7.1641020e-11 5.3043053e-03]
NumPy abs(z*conj(z)): [7.1641020e-11 5.3043053e-03]
JAX real(z*conj(z)): [7.1641026e-11 5.3043049e-03]
NumPy matches retained-real control: True
NumPy matches separate-squares control: False
JAX matches separate-squares control: True
JAX matches retained-real control: False
Separate abs(z)**2 control: [7.1641026e-11 5.3043053e-03]


## Hold periodograms while comparing averaging and bias division

An odd median selects an existing sorted value; an even median additionally averages its middle two values. Mean reduction and middle-value averaging are separate rounding boundaries. These fixed nonnegative arrays remove window and FFT differences.

The supplied positive factor `319/420` isolates bias division. Dividing once differs from multiplying by a reciprocal already rounded to float32. Widened division followed by a float32 cast is another control; the printed comparisons make no universal equality claim.

In [5]:
periodograms = (0.3 + np.abs(rng.standard_normal((7, 4)))).astype(np.float32)
for count in (6, 7):
    fixed = periodograms[:count]
    for name in ("mean", "median"):
        cpu_average = getattr(np, name)(fixed, axis=0)
        device_average = np.asarray(jax.jit(
            lambda x: getattr(jnp, name)(x, axis=0))(on_device(fixed)))
        report(f"Fixed {count}-row {name}", cpu_average, device_average)

bias = np.float32(319 / 420)
numerator = np.median(periodograms, axis=0)
direct_division = numerator / bias
rounded_reciprocal = np.float32(1.0) / bias
reciprocal_multiply = numerator * rounded_reciprocal
wide_division = np.float32(numerator.astype(np.float64) / np.float64(bias))
device_division = np.asarray(jax.jit(lambda x, b: x / b)(
    on_device(numerator), on_device(bias)))
print("Bias factor:", float(bias))
report("Bias division versus rounded-reciprocal multiply", direct_division, reciprocal_multiply)
report("Bias division versus widened-then-cast division", direct_division, wide_division)
report("Bias division versus device division", direct_division, device_division)


Fixed 6-row mean: differing values=1/4; max absolute difference=1.1920929e-07
Fixed 6-row median: differing values=0/4; max absolute difference=0
Fixed 7-row mean: differing values=1/4; max absolute difference=5.96046448e-08
Fixed 7-row median: differing values=0/4; max absolute difference=0
Bias factor: 0.7595238089561462
Bias division versus rounded-reciprocal multiply: differing values=2/4; max absolute difference=5.96046448e-08
Bias division versus widened-then-cast division: differing values=0/4; max absolute difference=0
Bias division versus device division: differing values=3/4; max absolute difference=1.1920929e-07


## Propagate final inverse-spectrum power into a reciprocal

After transforming an inverse-ASD response back to frequency, taking the reciprocal of its power makes earlier rounding visible in the PSD. This example starts at that held power boundary; it does not implement truncation, tapering or transform normalization.

The two supplied power expansions yield different float32 reciprocals. Direct device division and widened division are reported separately, so reciprocal rounding is not silently assigned to the power step.

In [6]:
separate_psd = np.float32(1.0) / separate_squares
retained_psd = np.float32(1.0) / retained_real_square
assert separate_psd.tobytes() != retained_psd.tobytes()
device_reciprocal = np.asarray(jax.jit(lambda p: 1.0 / p)(
    on_device(separate_squares)))
wide_reciprocal = np.float32(1.0 / separate_squares.astype(np.float64))
print("Reciprocal after separate squares:", separate_psd)
print("Reciprocal after retaining the real square:", retained_psd)
report("Same power: CPU versus device reciprocal", separate_psd, device_reciprocal)
report("Same power: CPU versus widened-then-cast reciprocal", separate_psd, wide_reciprocal)


Reciprocal after separate squares: [1.3958482e+10 1.8852612e+02]
Reciprocal after retaining the real square: [1.3958484e+10 1.8852609e+02]
Same power: CPU versus device reciprocal: differing values=1/2; max absolute difference=1024
Same power: CPU versus widened-then-cast reciprocal: differing values=0/2; max absolute difference=0


These controls distinguish changed inputs from changed rounding inside an operation. Fix the window before comparing transforms, fix the spectrum before comparing power, and fix periodograms or power before comparing reductions or reciprocals. Which stage differs remains a property of the actual implementation and supplied inputs; use the [PyCBC comparison notebook](jax_psd_numerical_differences.ipynb) for its original-implementation validation controls.